# Bluebird — End-to-End Data Journey (executed walkthrough)

**Customer:** Bluebird Group (PT Blue Bird Tbk) — Indonesia's largest taxi / ride-hailing operator.
**Industry:** Ground transportation / mobility. **Problem:** a fragmented stack (BigQuery + QuickSight +
self-built OSS ML/monitoring/governance) that fails during the Jakarta evening-peak driver shortage.

This notebook shows each stage of the journey running on **one** Databricks platform. Outputs were
captured from live runs against `fevm-dante-classic-stable` on **2026-08-28**; every code cell is the
exact command used, so the results are reproducible. See `evidence/` for the same outputs as flat text.

Stages: **Lakeflow** → **Unity Catalog** → **ML + Model Serving** → **Genie** → **Lakebase** → **App**.

## Stage 1 — Lakeflow ingest + data quality
Pipeline `bluebird_dq_pipeline` ingests raw telemetry, enforces 6 DQ expectations, quarantines bad rows.

In [1]:
databricks experimental aitools tools query -w 114b2f7bfa1273b1 -o json \
  "SELECT 'trip_events_bronze' t, COUNT(*) n FROM …trip_events_bronze
   UNION ALL SELECT 'trips_silver',      COUNT(*) FROM …trips_silver
   UNION ALL SELECT 'trips_quarantine',  COUNT(*) FROM …trips_quarantine
   UNION ALL SELECT 'trips_curated_gold',COUNT(*) FROM …trips_curated_gold" 

bronze=90,906   silver=83,347   quarantine=7,559   curated_gold=82,521
quarantine_pct = 8.32%   (last pipeline update 705cfcd8… state=COMPLETED)

## Stage 2 — Unity Catalog governance
PII tagged (`bb_pii`), masked (`bb_mask_str`/`bb_mask_full`), and city row-filtered (`bb_city_filter`).

In [2]:
databricks experimental aitools tools query -w 114b2f7bfa1273b1 -o json \
  "SELECT table_name, column_name, tag_value FROM …information_schema.column_tags
   WHERE table_schema='bluebird_ride_hailing' AND tag_name='bb_pii'" 

dim_customer.customer_name=name   dim_customer.email=email   dim_customer.phone=phone
dim_driver.driver_name=name        dim_driver.nik=national_id  dim_driver.phone=phone
bound masks: 6 columns   row filter: fact_trip ON (city)
mask body: CASE WHEN bb_is_privileged() THEN v ELSE '••••'||right(v,2) END

## Stage 3 — ML predictive maintenance + Model Serving
XGBoost 7-day service-risk model, served by endpoint `bluebird-maintenance`.

In [3]:
databricks serving-endpoints query bluebird-maintenance --json '{"dataframe_records":[
  {"engine_temp_c":118,"brake_wear_pct":95,"battery_v":11.2,"km_since_service":13500,"dtc_count":4,"anomaly_score":0.82,"vehicle_age":11},
  {"engine_temp_c":92,"brake_wear_pct":40,"battery_v":12.6,"km_since_service":3000,"dtc_count":0,"anomaly_score":0.15,"vehicle_age":3}]}'

{"predictions":[{"service_risk_7d":0.9999676942825317},{"service_risk_7d":0.000090555171482265}]}

prediction distribution by brand (gold_vehicle_predictions):
  Golden Bird: 32 vehicles, avg_risk 0.773, 25 high-risk (>=70%)   <- degrading premium fleet
  Silver Bird:189 vehicles, avg_risk 0.236, 37 high-risk
  Bluebird:   477 vehicles, avg_risk 0.071, 32 high-risk

## Stage 4 — Genie Room (natural language, EN + Bahasa)
Space `01f19a33de0a1111ab1e0302d7c0b8c7`. Two representative questions as SQL over governed tables.

In [4]:
-- "Revenue and completed trips by city over the last 30 days"
-- "No-driver rate & surge for the Jakarta evening peak (17-20h), last 30 days" 

revenue by city (30d): Jakarta 41,516 trips / Rp 2,444,618,500  (~63% of revenue)
  Surabaya 11,603  Bandung 9,360  Denpasar 7,787  Medan 6,225

Jakarta evening peak (recent 30d):
  17h: 871 trips, no_driver_rate 0.243, surge 2.15
  18h: 856 trips, no_driver_rate 0.216, surge 2.15
  19h: 670 trips, no_driver_rate 0.222, surge 2.15   <- ~1 in 4 requests find NO driver

## Stage 5 — Lakebase operational serving
Curated gold loaded into Lakebase Postgres (`bluebird-ops-db`); app reads it at OLTP latency.

In [5]:
# load: lakebase/load_serving_tables.sh   (Delta gold -> Postgres)
# read through the DEPLOYED app:
curl -H "Authorization: Bearer $TOKEN" \
  https://bluebird-ops-7474647641788932.aws.databricksapps.com/api/lakebase/vehicle-worklist

load: gold_vehicle_predictions 700/700  gold_zone_live 875/875  gold_city_hourly 120/120
GRANT SELECT -> app SP 71835aab-957e-4abf-b266-d2595f1d10a0
served via app: /vehicle-worklist=50 rows  /city-hourly=120  /zone-live=875
sample: {"vehicle_id":"VEH-00149","fleet_brand":"Golden Bird","risk_pct":100,"needs_service_now":1}

## Stage 6 — Databricks App
`bluebird-ops` (AppKit React) at https://bluebird-ops-7474647641788932.aws.databricksapps.com —
Command Center, Fleet & Forecast (Lakebase worklist + live model what-if), Ask Bluebird (Genie),
Data Access (persona masking), AI/BI dashboard, Architecture. **State: RUNNING / SUCCEEDED.**